# graph-vul: UniXcoder Fine-tuning on Google Colab

This notebook runs the full fine-tuning pipeline for function-level
vulnerability detection using UniXcoder, with Google Drive for persistent storage.

The current pipeline trains on the full labeled function subset across all CWEs
present in `func_subset.csv`. Projects touching held-out CWEs are routed into
`unseen_val` and `unseen_test` by `func_splits/assignments.csv`.

Optionally, set `USE_GRAPH = True` to fuse a lightweight RGCN graph embedding
(AST+CFG structure) alongside the code and CWE embeddings.

## Prerequisites

Before running, upload the following to **My Drive/graph-vul/data/**:

```
My Drive/
  graph-vul/
    data/
      func_subset.csv
      func_splits/
        assignments.csv
      processed/
        cwe_embeddings/
          <one .pt file for every CWE referenced by func_subset.csv>
        graph_artifacts/          # only if USE_GRAPH = True
          node_type_vocab.json
          edge_type_vocab.json
          <one .pt file per sample_id>
```

The notebook validates that:
- `assignments.csv` contains `train`, `val`, `test`, `unseen_val`, `unseen_test`
- every CWE referenced by `func_subset.csv` has a matching embedding file in
  `processed/cwe_embeddings/`
- (when `USE_GRAPH = True`) graph vocab files and artifacts exist

The `hf_cache/` directory is created automatically on first run to cache the
microsoft/unixcoder-base model weights (~500MB). Subsequent runs skip the download.

**Outputs** (written to Drive):
- `data/func_finetune_model.ckpt` — best model checkpoint (selected by val_f1)
- `data/func_finetune_metrics.json` — train/val/test + unseen_val/unseen_test metrics, including `*_0_64_f1` short-function metrics that are also logged to W&B

In [ ]:
import subprocess

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU."
    )
print(gpu_info.stdout)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT = "/content/drive/MyDrive/data/graph-vul"
DRIVE_DATA = f"{DRIVE_PROJECT}/data"
DRIVE_HF_CACHE = f"{DRIVE_PROJECT}/hf_cache"

In [17]:
!mkdir -p {DRIVE_DATA}
!mkdir -p {DRIVE_HF_CACHE}

In [ ]:
import csv
from pathlib import Path

data_root = Path(DRIVE_DATA)
subset_path = data_root / "func_subset.csv"
assignments_path = data_root / "func_splits" / "assignments.csv"
embeddings_dir = data_root / "processed" / "cwe_embeddings"

required_files = [subset_path, assignments_path, embeddings_dir]
missing = [str(path.relative_to(data_root)) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required paths in {DRIVE_DATA}:\n"
        + "\n".join(f"  - {path}" for path in missing)
    )

required_splits = {"train", "val", "test", "unseen_val", "unseen_test"}
with assignments_path.open(newline="") as handle:
    reader = csv.DictReader(handle)
    observed_splits = {row["split"] for row in reader if row.get("split")}

missing_splits = sorted(required_splits - observed_splits)
if missing_splits:
    raise ValueError(
        "assignments.csv is missing required splits:\n"
        + "\n".join(f"  - {split}" for split in missing_splits)
    )

with subset_path.open(newline="") as handle:
    reader = csv.DictReader(handle)
    cwe_ids = sorted({row["cwe_id"].strip() for row in reader if row.get("cwe_id", "").strip()})

missing_embeddings = [cwe_id for cwe_id in cwe_ids if not (embeddings_dir / f"{cwe_id}.pt").exists()]
if missing_embeddings:
    raise FileNotFoundError(
        "Missing CWE embeddings for func_subset.csv entries:\n"
        + "\n".join(f"  - processed/cwe_embeddings/{cwe_id}.pt" for cwe_id in missing_embeddings)
    )

print(f"Validated dataset root: {data_root}")
print(f"Found {len(cwe_ids)} CWE embedding files referenced by func_subset.csv.")
print(f"Observed splits: {sorted(observed_splits)}")

In [ ]:
# Validate graph artifacts (only checked when USE_GRAPH is enabled).
# This cell reads USE_GRAPH from the config cell below — re-run if you change it.
try:
    _use_graph = USE_GRAPH
except NameError:
    _use_graph = False

if _use_graph:
    graph_dir = data_root / "processed" / "graph_artifacts"
    node_vocab = graph_dir / "node_type_vocab.json"
    edge_vocab = graph_dir / "edge_type_vocab.json"

    missing_graph = []
    if not graph_dir.exists():
        missing_graph.append(str(graph_dir.relative_to(data_root)))
    if not node_vocab.exists():
        missing_graph.append(str(node_vocab.relative_to(data_root)))
    if not edge_vocab.exists():
        missing_graph.append(str(edge_vocab.relative_to(data_root)))
    if missing_graph:
        raise FileNotFoundError(
            f"USE_GRAPH=True but missing graph artifacts in {DRIVE_DATA}:\n"
            + "\n".join(f"  - {p}" for p in missing_graph)
        )

    # Count how many samples have graph artifacts
    graph_pts = list(graph_dir.glob("*.pt"))
    print(f"Graph mode enabled: {len(graph_pts)} graph artifacts found in {graph_dir}")
else:
    print("Graph mode disabled (USE_GRAPH=False). Skipping graph validation.")

In [20]:
import os

os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = DRIVE_HF_CACHE

In [ ]:
!git clone https://github.com/noamdwc/graph-vul.git /content/graph-vul
%pip install -q pytorch-lightning transformers sentencepiece pandas numpy \
    scikit-learn torchmetrics wandb torch-geometric

In [ ]:
import os
import shutil
from pathlib import Path

repo_data = "/content/graph-vul/data"

if os.path.islink(repo_data):
    os.unlink(repo_data)
elif os.path.isdir(repo_data):
    shutil.rmtree(repo_data)

os.symlink(DRIVE_DATA, repo_data)
assert Path(repo_data, "func_subset.csv").exists(), "Symlink verification failed"
print(f"Symlinked {repo_data} -> {DRIVE_DATA}")

## Weights & Biases

The training script logs to WandB. Run the cell below to authenticate.
To skip WandB logging, uncomment the `WANDB_MODE` line instead.

In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
!wandb login

## Training Configuration

Adjust hyperparameters in the cell below. Defaults match `graph_vul/config.py`.

| Parameter | Default | Notes |
|-----------|---------|-------|
| `NUM_EPOCHS` | 10 | |
| `BATCH_SIZE` | 8 | Per-GPU batch size |
| `ENCODER_LR` | 2e-5 | UniXcoder backbone learning rate |
| `HEAD_LR` | 1e-3 | Classification head learning rate |
| `GRAD_ACCUM` | 8 | Effective batch = BATCH_SIZE x GRAD_ACCUM = 64 |
| `SEED` | 42 | |
| `USE_GRAPH` | True | Enable RGCN graph (AST+CFG) features |
| `FUSION_LAYERS` | `list(range(12))` | All-layer gated residual fusion (E3 config) |
| `FINAL_GRAPH_CONCAT` | True | Late fusion: concatenate graph embedding before classifier |

**OOM?** Reduce `BATCH_SIZE` to 4 and increase `GRAD_ACCUM` to 16 to keep effective batch size at 64.

In [ ]:
NUM_EPOCHS = 10
BATCH_SIZE = 64
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
GRAD_ACCUM = 8
SEED = 42
USE_GRAPH = True
FUSION_LAYERS = list(range(12))  # all-layer gated residual fusion (E3)
FINAL_GRAPH_CONCAT = True

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

sys.path.insert(0, "/content/graph-vul")

from graph_vul.train_finetune import train

train(
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    encoder_lr=ENCODER_LR,
    head_lr=HEAD_LR,
    grad_accum=GRAD_ACCUM,
    seed=SEED,
    use_graph=USE_GRAPH,
    fusion_layers=FUSION_LAYERS,
    final_graph_concat=FINAL_GRAPH_CONCAT,
)

In [ ]:
import json
from pathlib import Path

metrics_path = Path(DRIVE_DATA) / "func_finetune_metrics.json"
ckpt_path = Path(DRIVE_DATA) / "func_finetune_model.ckpt"

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)

    groups = {
        "Seen CWE Metrics": ["train", "val", "test"],
        "Unseen CWE Metrics": ["unseen_val", "unseen_test"],
    }
    priority_suffixes = ["accuracy", "precision", "recall", "f1", "0_64_f1"]

    def ordered_keys(prefix):
        present = [k for k in metrics if k.startswith(f"{prefix}_")]
        ordered = [f"{prefix}_{suffix}" for suffix in priority_suffixes if f"{prefix}_{suffix}" in metrics]
        extras = sorted(k for k in present if k not in ordered)
        return ordered + extras

    for heading, prefixes in groups.items():
        print(f"=== {heading} ===")
        for prefix in prefixes:
            keys = ordered_keys(prefix)
            if keys:
                vals = "  ".join(f"{k}: {metrics[k]:.4f}" for k in keys)
                print(f"  {vals}")
                short_key = f"{prefix}_0_64_f1"
                if short_key in metrics:
                    print(f"    short functions (0-64 tokens) F1: {metrics[short_key]:.4f}")
            else:
                print(f"  {prefix}: no metrics found")
        print()
else:
    print(f"Metrics file not found at {metrics_path}")

if ckpt_path.exists():
    size_mb = ckpt_path.stat().st_size / (1024 * 1024)
    print(f"\nCheckpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
else:
    print(f"\nCheckpoint not found at {ckpt_path}")